In [75]:
# 데이터 처리 및 분석
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝 
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, r2_score, mean_squared_error, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier, XGBRegressor
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 시드 설정
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [76]:
df = pd.read_csv('data/merged_final_data.csv')

In [77]:
cols = df.columns
cols 

Index(['index', 'order_id', 'customer_id', 'customer_unique_id',
       'customer_zip_code_prefix', 'customer_city', 'customer_state',
       'customer_lat', 'customer_lng', 'total_items_count', 'seller_id',
       'price', 'freight_value', 'review_score', 'category',
       'seller_zip_code_prefix', 'seller_city', 'seller_state', 'seller_lat',
       'seller_lng', 'order_purchase_dayofweek', 'order_purchase_month',
       'approved_days', 'dispatch_days', 'delivery_days',
       'expected_delivery_days', 'delay_days', 'delay_days_int', 'is_delayed',
       'delay_days_cat', 'main_category', 'sub_category'],
      dtype='str')

In [78]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32758 entries, 0 to 32757
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   index                     32758 non-null  int64  
 1   order_id                  32758 non-null  str    
 2   customer_id               32758 non-null  str    
 3   customer_unique_id        32758 non-null  str    
 4   customer_zip_code_prefix  32758 non-null  float64
 5   customer_city             32758 non-null  str    
 6   customer_state            32758 non-null  str    
 7   customer_lat              32740 non-null  float64
 8   customer_lng              32740 non-null  float64
 9   total_items_count         32758 non-null  float64
 10  seller_id                 32758 non-null  str    
 11  price                     32758 non-null  float64
 12  freight_value             32758 non-null  float64
 13  review_score              32758 non-null  int64  
 14  category         

In [79]:
df.isna().sum()

index                         0
order_id                      0
customer_id                   0
customer_unique_id            0
customer_zip_code_prefix      0
customer_city                 0
customer_state                0
customer_lat                 18
customer_lng                 18
total_items_count             0
seller_id                     0
price                         0
freight_value                 0
review_score                  0
category                    452
seller_zip_code_prefix        0
seller_city                   0
seller_state                  0
seller_lat                    0
seller_lng                    0
order_purchase_dayofweek      0
order_purchase_month          0
approved_days                 0
dispatch_days                 0
delivery_days                 0
expected_delivery_days        0
delay_days                    0
delay_days_int                0
is_delayed                    0
delay_days_cat                0
main_category                 0
sub_cate

### 위도 경도 활용해서 거리 파생변수 생성

In [80]:
# Haversine 거리 계산 함수 (지구 곡률을 반영한 두 좌표 간의 직선 거리 km)
def haversine_vectorize(lat1, lon1, lat2, lon2):
    # 라디안 변환
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6371 * c # 6371: 지구의 평균 반지름(km)
    return km

# 거리(distance_km) 파생 변수 생성
df['distance_km'] = haversine_vectorize(
    df['customer_lat'], df['customer_lng'],
    df['seller_lat'], df['seller_lng']
)

#### 거리 파생변수 결측치 처리

In [81]:
print(f"distance_km 결측치 처리 전 행 수 : {len(df)}") # 95784
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median()) # 거리 중앙값으로 대체
print(f"distance_km 결측치 처리 후 행 수 : {len(df)}") # 95784

# 모델 학습에 불필요한 중간 컬럼 삭제
cols_to_drop = ['customer_zip_code_prefix', 'seller_zip_code_prefix', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng']
df = df.drop(columns=cols_to_drop)

distance_km 결측치 처리 전 행 수 : 32758
distance_km 결측치 처리 후 행 수 : 32758


In [82]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32758 entries, 0 to 32757
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   index                     32758 non-null  int64  
 1   order_id                  32758 non-null  str    
 2   customer_id               32758 non-null  str    
 3   customer_unique_id        32758 non-null  str    
 4   customer_city             32758 non-null  str    
 5   customer_state            32758 non-null  str    
 6   total_items_count         32758 non-null  float64
 7   seller_id                 32758 non-null  str    
 8   price                     32758 non-null  float64
 9   freight_value             32758 non-null  float64
 10  review_score              32758 non-null  int64  
 11  category                  32306 non-null  str    
 12  seller_city               32758 non-null  str    
 13  seller_state              32758 non-null  str    
 14  order_purchase_da

In [83]:
df.isna().sum()

index                         0
order_id                      0
customer_id                   0
customer_unique_id            0
customer_city                 0
customer_state                0
total_items_count             0
seller_id                     0
price                         0
freight_value                 0
review_score                  0
category                    452
seller_city                   0
seller_state                  0
order_purchase_dayofweek      0
order_purchase_month          0
approved_days                 0
dispatch_days                 0
delivery_days                 0
expected_delivery_days        0
delay_days                    0
delay_days_int                0
is_delayed                    0
delay_days_cat                0
main_category                 0
sub_category                  0
distance_km                   0
dtype: int64

### 머신러닝 모델 생성시 절대 안쓸만한 컬럼 제거

In [84]:
cols_to_drop = ['index','order_id','customer_id','customer_city','customer_state','seller_id',
                'category','seller_city','seller_state','delay_days_int']
df = df.drop(columns=cols_to_drop)
df['delay_days'] = df['delay_days'].astype(int) # delay_days 정수화

In [85]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 32758 entries, 0 to 32757
Data columns (total 17 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_unique_id        32758 non-null  str    
 1   total_items_count         32758 non-null  float64
 2   price                     32758 non-null  float64
 3   freight_value             32758 non-null  float64
 4   review_score              32758 non-null  int64  
 5   order_purchase_dayofweek  32758 non-null  str    
 6   order_purchase_month      32758 non-null  int64  
 7   approved_days             32758 non-null  int64  
 8   dispatch_days             32758 non-null  int64  
 9   delivery_days             32758 non-null  int64  
 10  expected_delivery_days    32758 non-null  int64  
 11  delay_days                32758 non-null  int64  
 12  is_delayed                32758 non-null  int64  
 13  delay_days_cat            32758 non-null  str    
 14  main_category    

In [86]:
df.isna().sum()

customer_unique_id          0
total_items_count           0
price                       0
freight_value               0
review_score                0
order_purchase_dayofweek    0
order_purchase_month        0
approved_days               0
dispatch_days               0
delivery_days               0
expected_delivery_days      0
delay_days                  0
is_delayed                  0
delay_days_cat              0
main_category               0
sub_category                0
distance_km                 0
dtype: int64

In [87]:
df.to_csv('./data/ml_data.csv')